# tilesparse — bulutta bir nokta

Bu defter **ana projeye dokunmuyor**: koşan şey `experiments/m1_run.py`, yerelde
koşanın aynısı. `cloud/` yalnız ücretsiz bir oturumun ihtiyacı olanı ekliyor —
duvar saati bütçesi ve varsayılan devam etme.

Hücreleri sırayla koşturun. **3. hücreyi atlamayın**: bu projedeki her zamanlama
sabiti "bu makinede ölçüldü" diyor ve bu makine o makine değil.

Ayrıntı, disk matematiği ve nokta bölüşümü için `cloud/README.md`.


## 1 — Platformu tanı ve depoyu getir

`REPO_URL`'i kendi uzak deponuzla değiştirin. Depo özel ise Kaggle'da Secrets,
Colab'da bir token gerekir; genel ise hiçbir kimlik bilgisi gerekmiyor.

Model **NousResearch aynası** — kapılı değil, yani HuggingFace token'ı da gerekmiyor.


In [ ]:
REPO_URL = 'https://github.com/KULLANICI/tilesparse'   # <-- degistirin

import os, sys, subprocess, shutil, pathlib

if pathlib.Path('/kaggle').exists():
    PLATFORM, ROOT = 'kaggle', pathlib.Path('/kaggle/working')
elif 'google.colab' in sys.modules or pathlib.Path('/content').exists():
    PLATFORM, ROOT = 'colab', pathlib.Path('/content')
else:
    PLATFORM, ROOT = 'local', pathlib.Path.cwd()

REPO = ROOT / 'tilesparse'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print(PLATFORM, '->', REPO)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'cloud/requirements.txt'], check=True)

import torch
p = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print('gpu:', p.name if p else 'YOK -- GPU calisma zamanini acin',
      f'{p.total_memory / 2**30:.1f} GiB' if p else '')


## 2 — Checkpoint nereye gidecek

**Bir nokta ~13 GiB tutuyor** ve blok dosyalarının hepsi sonuna kadar gerekli:
değerlendirme birleştirilmiş sıkıştırılmış model üzerinde koşuyor. Disk yetmezse
koşu *geç* çuvallar.

- **Kaggle**: `/kaggle/working` (20 GB) — sığar. Oturumlar arası kalıcılık için
  notebook'u *Save Version* ile koşturun.
- **Colab**: ücretsiz Drive 15 GB ve modelle birlikte **yetmez**. Ucuz noktaları
  (T=8, 16, 32, max) yerel diske yazıp tek oturumda bitirin; yalnız sonucu
  Drive'a kopyalayın.


In [ ]:
USE_DRIVE = False        # Colab'da True yaparsaniz Drive baglanir (yavas)

if PLATFORM == 'colab' and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESUME = pathlib.Path('/content/drive/MyDrive/tilesparse/resume')
else:
    RESUME = ROOT / 'resume'
RESUME.mkdir(parents=True, exist_ok=True)

free = shutil.disk_usage(RESUME).free / 2**30
print(f'{RESUME}  {free:.1f} GiB bos  (bir nokta ~13 istiyor)')


## 3 — Uçuş öncesi

Ortamı, diski ve hattın kendisini sınar; sonra **çekirdek hızlarını bu makinede
yeniden ölçer**. Sonunda, yeniden ölçmediği ve yeni bir kartın geçersiz
kılabileceği beş eşiği basar — `docs/STATUS.md` §6.13 tam olarak o hatanın
ne kadara mal olduğunu anlatıyor.


In [ ]:
!python cloud/preflight.py --resume-root {RESUME}


## 4 — Bir nokta koştur

Tasarım F'in yedi noktası **bağımsız**; her makineye birini (veya birkaçını)
verin. Modelin dediği süreler: T=1 4.01s, T=2 3.34, T=4 2.07, T=8 1.54,
T=16 1.14, T=32 0.92, T=max 0.66.

`--hours` oturum sınırının biraz altında olmalı. Bütçe dolarsa süreç **42** ile
çıkar ve checkpoint tamdır — aynı hücreyi yeniden koşturun, kaldığı yerden devam
eder.

`--calib-seqlen 2048`, `m1_run.py`'nin 4096 varsayılanı değil: C4 belgelerinin
yalnız %0.33'ü 4096 token'ı aşıyor ve örnekleyicinin deneme bütçesi yetişmiyor.
Gerekçe `cloud/README.md`'de.


In [ ]:
TILE  = '16'      # 1, 2, 4, 8, 16, 32, max
HOURS = 11.0 if PLATFORM == 'kaggle' else 3.5

import subprocess, sys
cmd = [sys.executable, '-u', 'cloud/run_point.py',
       '--tile', TILE, '--budget', '1.5', '--draw', '0',
       '--resume-root', str(RESUME), '--hours', str(HOURS),
       '--calib-samples', '128', '--calib-seqlen', '2048',
       '--datasets', 'wikitext2']
code_ = subprocess.run(cmd).returncode
print({0: 'BITTI', 42: 'BUTCE DOLDU -- bu hucreyi tekrar kosturun'}
      .get(code_, f'HATA (exit {code_})'))


## 5 — Sonucu al

Bitmiş bir nokta `<resume-root>/<slug>.json` dosyasına yazılıyor: perplexity,
katman başına göreli hata ve SNR, blok 0'ın yoğun E8P referansı (§3.2'nin
erken-uyarı kuralı), ve koşunun hangi kaldıraçlarla alındığı.


In [ ]:
import json, glob
for f in sorted(glob.glob(str(RESUME / '*.json'))):
    d = json.load(open(f))
    if 'perplexity' not in d: continue
    print(f"{d['spec']['tile_size']:>4}  {d['seconds']/3600:5.2f} h  "
          f"{d['perplexity']}  levers={d['levers']}")

if PLATFORM == 'colab' and pathlib.Path('/content/drive').exists():
    !mkdir -p /content/drive/MyDrive/tilesparse && cp {RESUME}/*.json /content/drive/MyDrive/tilesparse/
